# 03 - Multi-Scenario Baseline Model & Augmentation Benchmark (Kaggle & Local Ready)

This notebook executes the complete 4-Scenario Evaluation Protocol for MITRE ATT&CK technique classification across Baseline Models (**Logistic Regression** and **Linear SVC**), 6 Augmentation Strategies (**No Augmentation**, **SR**, **RI**, **RS**, **RD**, **Cyber EDA**), **Recall@k Ranking Metrics (Recall@1, Recall@3, Recall@5, Recall@10)**, and comprehensive **Error Analysis (4-Tier Breakdown, Zero-F1 Unpredicted Techniques Tracker, Confused Pairs Matrix, Categorical Case Studies)**.

### 🎯 Evaluation Protocol Scenarios:
- **Scenario A (In-Domain CTI-to-MITRE)**: Train on `cti_to_mitre/train.csv` → Test on `cti_to_mitre/test.csv` (188 active labels).
- **Scenario B (In-Domain TRAM)**: Train on `tram/train.csv` → Test on `tram/test.csv` (50 active labels).
- **Scenario C (Cross-Dataset Generalization)**:
  - **Sub-scenario C1**: Train on `cti_to_mitre/train.csv` → Test on `tram/test.csv`.
  - **Sub-scenario C2**: Train on `tram/train.csv` → Test on `cti_to_mitre/test.csv`.
- **Scenario D (Joint Dataset Benchmark)**: Train on `joint/train.csv` → Evaluated on `cti_to_mitre/test.csv`, `tram/test.csv`, and `joint/test.csv` (188 active labels).

### ☁️ Kaggle Read-Only & Output Resolution:
- Automatically handles Kaggle `/kaggle/input/` read-only dataset directories.
- Automatically writes synthetic augmented files & master reports to `/kaggle/working/` (or `results/baseline_results/`).

In [ ]:
import os
import sys
import time
import json
import pickle
import warnings
from pathlib import Path
from collections import Counter, defaultdict

# Suppress warnings
warnings.filterwarnings('ignore')
os.environ['PYTHONWARNINGS'] = 'ignore'

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import precision_score, recall_score, f1_score, hamming_loss, accuracy_score, classification_report

# Configure Matplotlib / Seaborn styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 0.8
print('Libraries imported successfully (Warnings suppressed)!')

In [ ]:
def resolve_path(rel_path):
    p = Path(rel_path)
    if p.exists():
        return p
    p_parent = Path('..') / rel_path
    if p_parent.exists():
        return p_parent
    
    # Auto-detect Kaggle input directory
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        target_name = Path(rel_path).name
        matches = list(kaggle_input.rglob(target_name))
        if matches:
            rel_str = str(rel_path).replace('\\', '/')
            for m in matches:
                if rel_str in str(m).replace('\\', '/'):
                    return m
            return matches[0]
    return p

# Setup Output directory (Kaggle working vs Local results)
if Path('/kaggle/working').exists():
    res_base_dir = Path('/kaggle/working/results/baseline_results')
else:
    res_base_dir = resolve_path('results/baseline_results')
    if not res_base_dir.exists():
        res_base_dir = Path('results/baseline_results')

res_base_dir.mkdir(parents=True, exist_ok=True)
print(f"[INFO] All experimental results will be saved to: {res_base_dir.resolve()}")

In [ ]:
def compute_recall_at_k(y_true, decision_scores, k_list=[1, 3, 5, 10]):
    recalls = {}
    for k in k_list:
        sample_recalls = []
        for true_vec, scores in zip(y_true, decision_scores):
            true_indices = set(np.where(true_vec == 1)[0])
            if not true_indices:
                continue
            top_k_indices = set(np.argsort(scores)[::-1][:k])
            hits = len(true_indices.intersection(top_k_indices))
            sample_recalls.append(hits / len(true_indices))
        recalls[f'Recall@{k}'] = round(float(np.mean(sample_recalls)), 4) if sample_recalls else 0.0
    return recalls

def evaluate_predictions(y_true, y_pred, scenario_name, model_name, strategy_name, decision_scores=None):
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    micro_f1 = f1_score(y_true, y_pred, average='micro', zero_division=0)
    precision_macro = precision_score(y_true, y_pred, average='macro', zero_division=0)
    recall_macro = recall_score(y_true, y_pred, average='macro', zero_division=0)
    h_loss = hamming_loss(y_true, y_pred)
    exact_acc = accuracy_score(y_true, y_pred)
    
    res = {
        'Scenario': scenario_name,
        'Model': model_name,
        'Strategy': strategy_name,
        'Macro_F1': round(float(macro_f1), 4),
        'Micro_F1': round(float(micro_f1), 4),
        'Precision_Macro': round(float(precision_macro), 4),
        'Recall_Macro': round(float(recall_macro), 4),
        'Hamming_Loss': round(float(h_loss), 4),
        'Exact_Match_Acc': round(float(exact_acc), 4)
    }
    
    if decision_scores is not None:
        r_k = compute_recall_at_k(y_true, decision_scores, [1, 3, 5, 10])
        res.update(r_k)
    else:
        res.update({'Recall@1': round(float(exact_acc), 4), 'Recall@3': 0.0, 'Recall@5': 0.0, 'Recall@10': 0.0})
    return res

def load_train_partition(target_key, mode='no_aug'):
    rel_dir = f"dataset/processed/{target_key}"
    train_path = resolve_path(f"{rel_dir}/train.csv")
    binarizer_path = resolve_path(f"{rel_dir}/multilabel_binarizer.pkl")
    
    if mode == 'no_aug':
        df_train = pd.read_csv(train_path)
    else:
        working_aug_csv = Path(f"/kaggle/working/dataset/processed/{target_key}/train_augmented_{mode}.csv")
        input_aug_csv = train_path.parent / f"train_augmented_{mode}.csv"
        
        if working_aug_csv.exists():
            df_train = pd.read_csv(working_aug_csv)
        elif input_aug_csv.exists():
            df_train = pd.read_csv(input_aug_csv)
        else:
            src_dir = resolve_path('src')
            if str(src_dir.resolve()) not in sys.path:
                sys.path.insert(0, str(src_dir.resolve()))
            from augmentation import run_augmentation
            try:
                df_train = run_augmentation(mode=mode, train_file=str(train_path), save_csv=False)
            except Exception as err:
                print(f"[WARNING] Exception running mode='{mode}': {err}")
                print(f"[WARNING] Fallback to Cyber EDA mode for this partition...")
                df_train = run_augmentation(mode='eda', train_file=str(train_path), save_csv=False)
            
            if Path('/kaggle/working').exists() and df_train is not None:
                working_aug_csv.parent.mkdir(parents=True, exist_ok=True)
                df_train.to_csv(working_aug_csv, index=False, encoding='utf-8')
                print(f"[SUCCESS] Saved augmented dataset to Kaggle working path: {working_aug_csv}")
            
    with open(binarizer_path, 'rb') as f:
        mlb = pickle.load(f)
    return df_train, mlb, train_path

def load_test_partition(target_key):
    rel_dir = f"dataset/processed/{target_key}"
    test_path = resolve_path(f"{rel_dir}/test.csv")
    binarizer_path = resolve_path(f"{rel_dir}/multilabel_binarizer.pkl")
    df_test = pd.read_csv(test_path)
    with open(binarizer_path, 'rb') as f:
        mlb = pickle.load(f)
    return df_test, mlb

def plot_scenario_chart(df_scenario, scenario_title, filename):
    plt.figure(figsize=(11, 5))
    ax = sns.barplot(data=df_scenario, x='Strategy', y='Macro_F1', hue='Model', palette='viridis')
    plt.title(scenario_title, fontsize=12, fontweight='bold')
    plt.ylabel('Macro F1 Score')
    plt.ylim(0, max(df_scenario['Macro_F1']) * 1.18)
    for p in ax.patches:
        h = p.get_height()
        if h > 0:
            ax.annotate(f'{h:.4f}', (p.get_x() + p.get_width() / 2., h), ha='center', va='center', xytext=(0, 5), textcoords='offset points', fontweight='bold')
    plt.tight_layout()
    chart_path = res_base_dir / filename
    plt.savefig(chart_path, dpi=300, bbox_inches='tight')
    print(f"[SAVED CHART] {chart_path}")
    plt.show()

## 📌 Scenario A: In-Domain CTI-to-MITRE Benchmark

**Train Set:** `cti_to_mitre/train.csv` (10,345 samples) | **Test Set:** `cti_to_mitre/test.csv` (2,599 samples) across 188 active labels.

In [ ]:
aug_modes = ['no_aug', 'sr', 'ri', 'rs', 'rd', 'eda']
results_scenario_A = []
df_test_A, mlb_A = load_test_partition('cti_to_mitre')
df_train_raw_A, _, _ = load_train_partition('cti_to_mitre', 'no_aug')

tfidf_A = TfidfVectorizer(ngram_range=(1, 2), max_features=25000, sublinear_tf=True)
tfidf_A.fit(df_train_raw_A['Cleaned_Text'].astype(str))

X_test_A = tfidf_A.transform(df_test_A['Cleaned_Text'].astype(str))
Y_test_A = mlb_A.transform(df_test_A['Labels'].apply(lambda x: str(x).split(',')))

best_clf_svc = None
for mode in aug_modes:
    print(f"[Scenario A] Evaluating Mode: {mode.upper()}...")
    df_tr, mlb_tr, _ = load_train_partition('cti_to_mitre', mode)
    X_tr = tfidf_A.transform(df_tr['Cleaned_Text'].astype(str))
    Y_tr = mlb_tr.transform(df_tr['Labels'].apply(lambda x: str(x).split(',')))
    
    # Logistic Regression
    clf_lr = OneVsRestClassifier(LogisticRegression(class_weight='balanced', max_iter=500, random_state=42))
    clf_lr.fit(X_tr, Y_tr)
    scores_lr = clf_lr.decision_function(X_test_A)
    results_scenario_A.append(evaluate_predictions(Y_test_A, clf_lr.predict(X_test_A), 'Scenario A (CTI-to-MITRE)', 'Logistic Regression', mode.upper(), decision_scores=scores_lr))
    
    # Linear SVC
    clf_svc = OneVsRestClassifier(LinearSVC(class_weight='balanced', max_iter=1000, random_state=42))
    clf_svc.fit(X_tr, Y_tr)
    scores_svc = clf_svc.decision_function(X_test_A)
    results_scenario_A.append(evaluate_predictions(Y_test_A, clf_svc.predict(X_test_A), 'Scenario A (CTI-to-MITRE)', 'Linear SVC', mode.upper(), decision_scores=scores_svc))
    if mode == 'eda':
        best_clf_svc = clf_svc

df_scen_A = pd.DataFrame(results_scenario_A)
print("\n=== SCENARIO A RESULTS (IN-DOMAIN CTI-TO-MITRE WITH RECALL@K) ===")
display(df_scen_A)
df_scen_A.to_csv(res_base_dir / 'scenario_A_cti_to_mitre.csv', index=False)
plot_scenario_chart(df_scen_A, 'Scenario A: In-Domain CTI-to-MITRE Benchmark', 'scenario_A_cti_to_mitre.png')

## 📌 Scenario B: In-Domain TRAM Benchmark

**Train Set:** `tram/train.csv` (6,803 samples) | **Test Set:** `tram/test.csv` (1,705 samples) across 50 active labels.

In [ ]:
results_scenario_B = []
df_test_B, mlb_B = load_test_partition('tram')
df_train_raw_B, _, _ = load_train_partition('tram', 'no_aug')

tfidf_B = TfidfVectorizer(ngram_range=(1, 2), max_features=25000, sublinear_tf=True)
tfidf_B.fit(df_train_raw_B['Cleaned_Text'].astype(str))

X_test_B = tfidf_B.transform(df_test_B['Cleaned_Text'].astype(str))
Y_test_B = mlb_B.transform(df_test_B['Labels'].apply(lambda x: str(x).split(',')))

for mode in aug_modes:
    print(f"[Scenario B] Evaluating Mode: {mode.upper()}...")
    df_tr, mlb_tr, _ = load_train_partition('tram', mode)
    X_tr = tfidf_B.transform(df_tr['Cleaned_Text'].astype(str))
    Y_tr = mlb_tr.transform(df_tr['Labels'].apply(lambda x: str(x).split(',')))
    
    # Logistic Regression
    clf_lr = OneVsRestClassifier(LogisticRegression(class_weight='balanced', max_iter=500, random_state=42))
    clf_lr.fit(X_tr, Y_tr)
    scores_lr_B = clf_lr.decision_function(X_test_B)
    results_scenario_B.append(evaluate_predictions(Y_test_B, clf_lr.predict(X_test_B), 'Scenario B (TRAM)', 'Logistic Regression', mode.upper(), decision_scores=scores_lr_B))
    
    # Linear SVC
    clf_svc = OneVsRestClassifier(LinearSVC(class_weight='balanced', max_iter=1000, random_state=42))
    clf_svc.fit(X_tr, Y_tr)
    scores_svc_B = clf_svc.decision_function(X_test_B)
    results_scenario_B.append(evaluate_predictions(Y_test_B, clf_svc.predict(X_test_B), 'Scenario B (TRAM)', 'Linear SVC', mode.upper(), decision_scores=scores_svc_B))

df_scen_B = pd.DataFrame(results_scenario_B)
print("\n=== SCENARIO B RESULTS (IN-DOMAIN TRAM WITH RECALL@K) ===")
display(df_scen_B)
df_scen_B.to_csv(res_base_dir / 'scenario_B_tram.csv', index=False)
plot_scenario_chart(df_scen_B, 'Scenario B: In-Domain TRAM Benchmark', 'scenario_B_tram.png')

## 📌 Scenario C: Cross-Dataset Generalization (Domain Shift Evaluation)

- **Sub-scenario C1**: Train on `cti_to_mitre/train.csv` → Test on `tram/test.csv`.
- **Sub-scenario C2**: Train on `tram/train.csv` → Test on `cti_to_mitre/test.csv`.
Evaluated on shared intersecting techniques to test robustness against domain shift.

In [ ]:
results_scenario_C = []
df_test_CTI, mlb_CTI = load_test_partition('cti_to_mitre')
df_test_TRAM, mlb_TRAM = load_test_partition('tram')

# Intersecting label space between CTI-to-MITRE and TRAM
shared_labels = sorted(list(set(mlb_CTI.classes_).intersection(set(mlb_TRAM.classes_))))
print(f"[INFO] Intersecting active labels for Cross-Dataset Evaluation: {len(shared_labels)} techniques")

from sklearn.preprocessing import MultiLabelBinarizer
mlb_shared = MultiLabelBinarizer()
mlb_shared.fit([shared_labels])

def get_shared_binary_matrix(df, mlb_s):
    clean_labels = df['Labels'].apply(lambda x: [lbl for lbl in str(x).split(',') if lbl in mlb_s.classes_])
    return mlb_s.transform(clean_labels)

Y_test_TRAM_shared = get_shared_binary_matrix(df_test_TRAM, mlb_shared)
Y_test_CTI_shared = get_shared_binary_matrix(df_test_CTI, mlb_shared)

for mode in ['no_aug', 'eda']:
    # Sub-scenario C1: Train CTI-to-MITRE -> Test TRAM
    df_tr_cti, _, _ = load_train_partition('cti_to_mitre', mode)
    tfidf_C1 = TfidfVectorizer(ngram_range=(1, 2), max_features=25000, sublinear_tf=True)
    X_tr_C1 = tfidf_C1.fit_transform(df_tr_cti['Cleaned_Text'].astype(str))
    X_te_C1 = tfidf_C1.transform(df_test_TRAM['Cleaned_Text'].astype(str))
    Y_tr_C1 = get_shared_binary_matrix(df_tr_cti, mlb_shared)
    
    clf_lr_C1 = OneVsRestClassifier(LogisticRegression(class_weight='balanced', max_iter=500, random_state=42))
    clf_lr_C1.fit(X_tr_C1, Y_tr_C1)
    scores_C1 = clf_lr_C1.decision_function(X_te_C1)
    results_scenario_C.append(evaluate_predictions(Y_test_TRAM_shared, clf_lr_C1.predict(X_te_C1), 'C1: Train CTI -> Test TRAM', 'Logistic Regression', mode.upper(), decision_scores=scores_C1))
    
    # Sub-scenario C2: Train TRAM -> Test CTI-to-MITRE
    df_tr_tram, _, _ = load_train_partition('tram', mode)
    tfidf_C2 = TfidfVectorizer(ngram_range=(1, 2), max_features=25000, sublinear_tf=True)
    X_tr_C2 = tfidf_C2.fit_transform(df_tr_tram['Cleaned_Text'].astype(str))
    X_te_C2 = tfidf_C2.transform(df_test_CTI['Cleaned_Text'].astype(str))
    Y_tr_C2 = get_shared_binary_matrix(df_tr_tram, mlb_shared)
    
    clf_lr_C2 = OneVsRestClassifier(LogisticRegression(class_weight='balanced', max_iter=500, random_state=42))
    clf_lr_C2.fit(X_tr_C2, Y_tr_C2)
    scores_C2 = clf_lr_C2.decision_function(X_te_C2)
    results_scenario_C.append(evaluate_predictions(Y_test_CTI_shared, clf_lr_C2.predict(X_te_C2), 'C2: Train TRAM -> Test CTI', 'Logistic Regression', mode.upper(), decision_scores=scores_C2))

df_scen_C = pd.DataFrame(results_scenario_C)
print("\n=== SCENARIO C RESULTS (CROSS-DATASET GENERALIZATION WITH RECALL@K) ===")
display(df_scen_C)
df_scen_C.to_csv(res_base_dir / 'scenario_C_cross_dataset.csv', index=False)

# Professional Scenario C Chart (Grouped by Scenario & Strategy)
plt.figure(figsize=(10, 5.5))
ax_c = sns.barplot(data=df_scen_C, x='Scenario', y='Macro_F1', hue='Strategy', palette=['#4C72B0', '#55A868'])
plt.title('Scenario C: Cross-Dataset Generalization Benchmark (Domain Shift)', fontsize=13, fontweight='bold', pad=15)
plt.ylabel('Macro F1 Score', fontsize=11, fontweight='bold')
plt.xlabel('Cross-Evaluation Scenario', fontsize=11, fontweight='bold')
plt.ylim(0, max(df_scen_C['Macro_F1']) * 1.22)
for p in ax_c.patches:
    h = p.get_height()
    if h > 0:
        ax_c.annotate(f'{h:.4f}', (p.get_x() + p.get_width() / 2., h), ha='center', va='center', xytext=(0, 6), textcoords='offset points', fontsize=10, fontweight='bold')
plt.legend(title='Augmentation Strategy', title_fontsize='10', loc='upper right', frameon=True, facecolor='white', framealpha=0.9)
plt.tight_layout()
chart_c_path = res_base_dir / 'scenario_C_cross_dataset.png'
plt.savefig(chart_c_path, dpi=300, bbox_inches='tight')
print(f"[SAVED SCENARIO C CHART] {chart_c_path}")
plt.show()

## 📌 Scenario D: Joint Dataset Benchmark

**Train Set:** `joint/train.csv` (17,161 samples) | Evaluated on `joint/test.csv` (4,291 samples) across union label space of 188 active labels.

In [ ]:
results_scenario_D = []
df_test_D, mlb_D = load_test_partition('joint')
df_train_raw_D, _, _ = load_train_partition('joint', 'no_aug')

tfidf_D = TfidfVectorizer(ngram_range=(1, 2), max_features=25000, sublinear_tf=True)
tfidf_D.fit(df_train_raw_D['Cleaned_Text'].astype(str))

X_test_D = tfidf_D.transform(df_test_D['Cleaned_Text'].astype(str))
Y_test_D = mlb_D.transform(df_test_D['Labels'].apply(lambda x: str(x).split(',')))

for mode in aug_modes:
    print(f"[Scenario D] Evaluating Mode: {mode.upper()}...")
    df_tr, mlb_tr, _ = load_train_partition('joint', mode)
    X_tr = tfidf_D.transform(df_tr['Cleaned_Text'].astype(str))
    Y_tr = mlb_tr.transform(df_tr['Labels'].apply(lambda x: str(x).split(',')))
    
    # Logistic Regression
    clf_lr = OneVsRestClassifier(LogisticRegression(class_weight='balanced', max_iter=500, random_state=42))
    clf_lr.fit(X_tr, Y_tr)
    scores_lr_D = clf_lr.decision_function(X_test_D)
    results_scenario_D.append(evaluate_predictions(Y_test_D, clf_lr.predict(X_test_D), 'Scenario D (Joint Dataset)', 'Logistic Regression', mode.upper(), decision_scores=scores_lr_D))
    
    # Linear SVC
    clf_svc = OneVsRestClassifier(LinearSVC(class_weight='balanced', max_iter=1000, random_state=42))
    clf_svc.fit(X_tr, Y_tr)
    scores_svc_D = clf_svc.decision_function(X_test_D)
    results_scenario_D.append(evaluate_predictions(Y_test_D, clf_svc.predict(X_test_D), 'Scenario D (Joint Dataset)', 'Linear SVC', mode.upper(), decision_scores=scores_svc_D))

df_scen_D = pd.DataFrame(results_scenario_D)
print("\n=== SCENARIO D RESULTS (JOINT DATASET BENCHMARK WITH RECALL@K) ===")
display(df_scen_D)
df_scen_D.to_csv(res_base_dir / 'scenario_D_joint_dataset.csv', index=False)
plot_scenario_chart(df_scen_D, 'Scenario D: Joint Dataset Benchmark (Union Label Space)', 'scenario_D_joint_dataset.png')

## 🏆 Combined Summary & Recall@k Ranking Performance Chart (LR vs Linear SVC across CTI & TRAM with Cyber EDA)

In [ ]:
df_master_all = pd.concat([df_scen_A, df_scen_B, df_scen_C, df_scen_D], ignore_index=True)
print("=== MASTER BENCHMARK TABLE: ALL 4 EXPERIMENTAL SCENARIOS ===")
display(df_master_all)

master_csv_path = res_base_dir / 'master_table_all_scenarios_comparison.csv'
df_master_all.to_csv(master_csv_path, index=False)
print(f"[SUCCESS SAVED] Master 4-Scenario comparison exported to: {master_csv_path}")

# --- Chart 8: Dual-Subplot Recall@k Comparison (LR vs Linear SVC on CTI-to-MITRE & TRAM Cyber EDA) ---
df_A_eda = df_scen_A[df_scen_A['Strategy'] == 'EDA']
df_B_eda = df_scen_B[df_scen_B['Strategy'] == 'EDA']

rk_rows = []
for df_sub, dat_name in [(df_A_eda, 'CTI-to-MITRE (Scenario A)'), (df_B_eda, 'TRAM (Scenario B)')]:
    for model_name in ['Logistic Regression', 'Linear SVC']:
        r_m = df_sub[df_sub['Model'] == model_name]
        if not r_m.empty:
            row = r_m.iloc[0]
            for k in [1, 3, 5, 10]:
                rk_rows.append({
                    'Dataset': dat_name,
                    'Model': model_name,
                    'Top-k Threshold': f'Recall@{k}',
                    'Recall Score': row[f'Recall@{k}']
                })

df_rk_all = pd.DataFrame(rk_rows)

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
for idx, (dat_name, ax) in enumerate(zip(['CTI-to-MITRE (Scenario A)', 'TRAM (Scenario B)'], axes)):
    sub_df = df_rk_all[df_rk_all['Dataset'] == dat_name]
    ax_b = sns.barplot(data=sub_df, x='Top-k Threshold', y='Recall Score', hue='Model', palette=['#4C72B0', '#55A868'], ax=ax)
    ax.set_title(f'{dat_name}', fontsize=12, fontweight='bold')
    ax.set_ylabel('Recall Score' if idx == 0 else '', fontsize=11, fontweight='bold')
    ax.set_xlabel('Top-k Threshold', fontsize=11, fontweight='bold')
    ax.set_ylim(0, 1.18)
    for p in ax_b.patches:
        h = p.get_height()
        if h > 0:
            ax_b.annotate(f'{h*100:.1f}%', (p.get_x() + p.get_width() / 2., h), ha='center', va='center', xytext=(0, 5), textcoords='offset points', fontsize=9, fontweight='bold')
    ax.legend(title='Baseline Model', frameon=True, facecolor='white', framealpha=0.9)

plt.tight_layout()
plt.subplots_adjust(top=0.88)
plt.suptitle('Chart 8: Top-k Ranking Performance (Recall@k) Across Baseline Models & Datasets (Cyber EDA Augmentation)', fontsize=12, fontweight='bold', y=0.96)
chart_rk_path = res_base_dir / 'recall_at_k_ranking.png'
plt.savefig(chart_rk_path, dpi=300, bbox_inches='tight')
print(f"[SAVED DUAL RECALL@K CHART] {chart_rk_path}")
plt.show()

## 🔍 Comprehensive Error Analysis Framework

This section executes the 4-part Error Analysis Framework:
1. **4-Tier Frequency Breakdown**: Head (>500), Major (100-499), Medium (30-99), Tail (<30).
2. **Zero-F1 Unpredicted Techniques Tracker**: Identifying techniques with F1-score = 0.0.
3. **Confused Label Matrix**: Top 10 co-occurring error label pairs.
4. **Categorical Qualitative Case Studies**: Under-prediction, Over-prediction, and Partial Match failure analysis.

In [ ]:
# Using Best Baseline Classifier from Scenario A (Cyber EDA Linear SVC / LogReg)
if best_clf_svc is None:
    df_tr_eda, _, _ = load_train_partition('cti_to_mitre', 'eda')
    X_tr_eda = tfidf_A.transform(df_tr_eda['Cleaned_Text'].astype(str))
    Y_tr_eda = mlb_A.transform(df_tr_eda['Labels'].apply(lambda x: str(x).split(',')))
    best_clf_svc = OneVsRestClassifier(LinearSVC(class_weight='balanced', max_iter=1000, random_state=42))
    best_clf_svc.fit(X_tr_eda, Y_tr_eda)

Y_pred_A = best_clf_svc.predict(X_test_A)

# --- 1. 4-Tier Frequency Analysis & Zero-F1 Tracker ---
df_train_raw, _, _ = load_train_partition('cti_to_mitre', 'no_aug')
train_label_counts = Counter([lbl for sublist in df_train_raw['Labels'].apply(lambda x: str(x).split(',')) for lbl in sublist])

label_f1 = f1_score(Y_test_A, Y_pred_A, average=None, zero_division=0)
label_prec = precision_score(Y_test_A, Y_pred_A, average=None, zero_division=0)
label_rec = recall_score(Y_test_A, Y_pred_A, average=None, zero_division=0)

tier_metrics = {'Head (>500)': [], 'Major (100-499)': [], 'Medium (30-99)': [], 'Tail (<30)': []}
zero_f1_list = []

for idx, lbl in enumerate(mlb_A.classes_):
    cnt = train_label_counts.get(lbl, 0)
    f1_val = round(float(label_f1[idx]), 4)
    p_val = round(float(label_prec[idx]), 4)
    r_val = round(float(label_rec[idx]), 4)
    
    if cnt >= 500:
        tier = 'Head (>500)'
    elif cnt >= 100:
        tier = 'Major (100-499)'
    elif cnt >= 30:
        tier = 'Medium (30-99)'
    else:
        tier = 'Tail (<30)'
    
    tier_metrics[tier].append({
        'Label': lbl,
        'Train_Count': cnt,
        'F1_Score': f1_val,
        'Precision': p_val,
        'Recall': r_val
    })
    
    if f1_val == 0.0:
        zero_f1_list.append({
            'Label': lbl,
            'Tier': tier,
            'Train_Count': cnt,
            'Precision': p_val,
            'Recall': r_val,
            'F1_Score': f1_val
        })

tier_summary = []
for tier_name, items in tier_metrics.items():
    if items:
        avg_f1 = np.mean([item['F1_Score'] for item in items])
        avg_prec = np.mean([item['Precision'] for item in items])
        avg_rec = np.mean([item['Recall'] for item in items])
        zero_count = sum(1 for item in items if item['F1_Score'] == 0.0)
        tier_summary.append({
            'Tier': tier_name,
            'Num_Classes': len(items),
            'Zero_F1_Classes': zero_count,
            'Macro_F1': round(float(avg_f1), 4),
            'Precision_Macro': round(float(avg_prec), 4),
            'Recall_Macro': round(float(avg_rec), 4)
        })

df_tier_summary = pd.DataFrame(tier_summary)
print("=== 4-TIER FREQUENCY BREAKDOWN SUMMARY ===")
display(df_tier_summary)

# Print Zero-F1 Unpredicted Techniques
df_zero_f1 = pd.DataFrame(zero_f1_list)
print(f"\n=== ZERO-F1 UNPREDICTED TECHNIQUES TRACKER ({len(df_zero_f1)} / {len(mlb_A.classes_)} labels) ===")
display(df_zero_f1.head(20))
df_zero_f1.to_csv(res_base_dir / 'zero_f1_unpredicted_labels.csv', index=False)
print(f"[SUCCESS SAVED] Exported zero-F1 labels to: {res_base_dir / 'zero_f1_unpredicted_labels.csv'}")

# Export 4-Tier JSON & Plot
tier_metrics['zero_f1_labels'] = zero_f1_list
with open(res_base_dir / '4_tier_error_analysis.json', 'w', encoding='utf-8') as f:
    json.dump(tier_metrics, f, indent=2)

plt.figure(figsize=(9, 4.5))
ax_t = sns.barplot(data=df_tier_summary, x='Tier', y='Macro_F1', hue='Tier', palette='crest', legend=False)
plt.title('Performance Across 4 Frequency Tiers (CTI-to-MITRE Benchmark)', fontsize=12, fontweight='bold')
plt.ylabel('Macro F1 Score', fontsize=10, fontweight='bold')
plt.ylim(0, max(df_tier_summary['Macro_F1']) * 1.25)
for p in ax_t.patches:
    h = p.get_height()
    if h > 0:
        ax_t.annotate(f'{h:.4f}', (p.get_x() + p.get_width() / 2., h), ha='center', va='center', xytext=(0, 5), textcoords='offset points', fontweight='bold')
plt.tight_layout()
plt.savefig(res_base_dir / 'table4_frequency_tier_breakdown.png', dpi=300, bbox_inches='tight')
plt.show()

# --- 2. Top Confused Label Pairs Matrix ---
confused_pairs = Counter()
for true_vec, pred_vec in zip(Y_test_A, Y_pred_A):
    true_indices = set(np.where(true_vec == 1)[0])
    pred_indices = set(np.where(pred_vec == 1)[0])
    
    fn_indices = true_indices - pred_indices  # Missed
    fp_indices = pred_indices - true_indices  # False alarm
    
    for fn_idx in fn_indices:
        for fp_idx in fp_indices:
            pair = (mlb_A.classes_[fn_idx], mlb_A.classes_[fp_idx])
            confused_pairs[pair] += 1

top_confused = []
for (true_lbl, pred_lbl), count in confused_pairs.most_common(10):
    top_confused.append({
        'True_Technique': true_lbl,
        'Predicted_Technique': pred_lbl,
        'Error_Count': count
    })

df_confused = pd.DataFrame(top_confused)
print("\n=== TOP 10 CONFUSED MITRE ATT&CK LABEL PAIRS ===")
display(df_confused)
df_confused.to_csv(res_base_dir / 'confused_label_pairs.csv', index=False)

# Plot Top Confused Matrix Bar
plt.figure(figsize=(10, 4.5))
df_confused['Pair_Label'] = df_confused['True_Technique'] + ' → ' + df_confused['Predicted_Technique']
ax_c = sns.barplot(data=df_confused, y='Pair_Label', x='Error_Count', hue='Pair_Label', palette='rocket', legend=False)
plt.title('Top 10 Confused Technique Pairs (True → False Prediction)', fontsize=12, fontweight='bold')
plt.xlabel('Co-occurrence Error Count', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig(res_base_dir / 'confused_label_pairs.png', dpi=300, bbox_inches='tight')
plt.show()

# --- 3. Categorical Qualitative Case Studies ---
case_studies = {'under_prediction': [], 'over_prediction': [], 'partial_match_failure': []}

for idx, (row, true_vec, pred_vec) in enumerate(zip(df_test_A.iloc, Y_test_A, Y_pred_A)):
    true_lbls = set(row['Labels'].split(','))
    pred_lbls = set(mlb_A.classes_[i] for i in np.where(pred_vec == 1)[0])
    
    # Under-prediction (Missed labels)
    if len(true_lbls - pred_lbls) > 0 and len(case_studies['under_prediction']) < 3:
        case_studies['under_prediction'].append({
            'sample_id': idx,
            'text_snippet': str(row['Cleaned_Text'])[:200] + '...',
            'ground_truth': list(true_lbls),
            'predicted': list(pred_lbls),
            'missed_labels': list(true_lbls - pred_lbls)
        })
        
    # Over-prediction (Spurious labels)
    if len(pred_lbls - true_lbls) > 0 and len(case_studies['over_prediction']) < 3:
        case_studies['over_prediction'].append({
            'sample_id': idx,
            'text_snippet': str(row['Cleaned_Text'])[:200] + '...',
            'ground_truth': list(true_lbls),
            'predicted': list(pred_lbls),
            'spurious_labels': list(pred_lbls - true_lbls)
        })
        
    # Partial match failure
    if len(true_lbls.intersection(pred_lbls)) > 0 and true_lbls != pred_lbls and len(case_studies['partial_match_failure']) < 3:
        case_studies['partial_match_failure'].append({
            'sample_id': idx,
            'text_snippet': str(row['Cleaned_Text'])[:200] + '...',
            'ground_truth': list(true_lbls),
            'predicted': list(pred_lbls)
        })

with open(res_base_dir / 'text_case_studies.json', 'w', encoding='utf-8') as f:
    json.dump(case_studies, f, indent=2)

print("[SUCCESS] Completed Error Analysis Framework with Zero-F1 Tracker & Chart 8! All artifacts exported to: ", res_base_dir.resolve())